<a href="https://colab.research.google.com/github/cnsalsabila/myXLSentimentAnalysis/blob/main/Week%202-3/Preprocessing_myXL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing Review myXL
1. Lowercasing
2. Punctuation Removal
3. Expand Contractions / Slang Normalization
4. Tokenization
5. Stopword Removal (Bahasa Indonesia)
6. Stemming (Sastrawi - Bahasa Indonesia)
8. Rare Words Removal
9. Common Words Removal

## Install dan Import Library yang Dibutuhkan

In [1]:
!pip install PySastrawi
!pip install nltk
!pip install pyspellchecker
!pip install contractions
!pip install pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.6/210.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from spellchecker import SpellChecker
from collections import Counter
from tqdm import tqdm

# Download resource NLTK
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

tqdm.pandas()
print('Semua library berhasil diimport!')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Semua library berhasil diimport!


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## Upload & Load Dataset

In [4]:
from google.colab import files
uploaded = files.upload()

Saving df_myxl_rev (1).csv to df_myxl_rev (1).csv


In [5]:
# Ambil nama file yang diupload
filename = list(uploaded.keys())[0]
print(f'File yang diupload: {filename}')

# Load dataset
df = pd.read_csv(filename)
print(f'Shape dataset: {df.shape}')
df.head()

File yang diupload: df_myxl_rev (1).csv
Shape dataset: (1006280, 11)


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,ef57d849-9936-43c6-a00f-ee67e1fb99c3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,MAKIN HARI BUKANNYA TAMBAH BAIK MALAH TAMBAH B...,1,0,8.4.0,2026-04-20 16:55:58,"Hi Kak, mohon maaf ya kalau ada yang bikin ga ...",2026-04-20 17:00:10,8.4.0
1,8f583ddf-e440-4531-b757-c6085f092385,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,signal jatingan buruk & harga paket internet m...,2,0,8.10.0,2026-04-20 16:55:23,"Hi Kak, mohon maaf atas kendala yang terjadi. ...",2026-04-20 17:00:12,8.10.0
2,07656f52-a153-4940-aaa3-44a291e328e2,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,pelayanan busuk.. masa daftar kuota di apk jad...,1,0,9.1.0,2026-04-20 16:54:47,"Hi Kak, maaf ya, agar dapat dibantu cek lebih ...",2026-04-20 17:00:13,9.1.0
3,64704030-3857-48cb-99f5-f695e09fa613,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,BUSUK KALI .. MAU NGECLAIM KUOTA AKTIVASI AJA ...,1,0,NaN,2026-04-20 16:52:52,"Hi Kak, mohon maaf ya atas ketidaknyamanan yan...",2026-04-20 17:00:15,NaN
4,4baaefe9-4c34-43dd-8483-58535b8b3ea3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"tolong min ditempat ku sinyal full, kuota bany...",1,0,9.1.0,2026-04-20 16:46:39,"Hi Kak, mohon maaf ya atas kendalanya. Yuk hub...",2026-04-20 17:00:17,9.1.0


In [6]:
# Cek info dataset
print(df.info())
print('\nJumlah missing value per kolom:')
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1006280 entries, 0 to 1006279
Data columns (total 11 columns):
 #   Column                Non-Null Count    Dtype 
---  ------                --------------    ----- 
 0   reviewId              1006280 non-null  object
 1   userName              1006280 non-null  object
 2   userImage             1006280 non-null  object
 3   content               1006266 non-null  object
 4   score                 1006280 non-null  int64 
 5   thumbsUpCount         1006280 non-null  int64 
 6   reviewCreatedVersion  817635 non-null   object
 7   at                    1006280 non-null  object
 8   replyContent          801590 non-null   object
 9   repliedAt             801590 non-null   object
 10  appVersion            817635 non-null   object
dtypes: int64(2), object(9)
memory usage: 84.5+ MB
None

Jumlah missing value per kolom:
reviewId                     0
userName                     0
userImage                    0
content                     1

In [7]:
# Ambil kolom 'content' saja sebagai teks review
# Drop baris yang kolom content-nya kosong
df_clean = df[['content', 'score']].copy()
df_clean.dropna(subset=['content'], inplace=True)
df_clean.reset_index(drop=True, inplace=True)

print(f'Jumlah data setelah hapus NaN: {len(df_clean)}')
df_clean.head()

Jumlah data setelah hapus NaN: 1006266


,content,score
0,MAKIN HARI BUKANNYA TAMBAH BAIK MALAH TAMBAH B...,1
1,signal jatingan buruk & harga paket internet m...,2
2,pelayanan busuk.. masa daftar kuota di apk jad...,1
3,BUSUK KALI .. MAU NGECLAIM KUOTA AKTIVASI AJA ...,1
4,"tolong min ditempat ku sinyal full, kuota bany...",1


## Inisialisasi Tools Preprocessing

In [8]:
# ----- Stemmer Bahasa Indonesia (Sastrawi) -----
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# ----- Stopwords Bahasa Indonesia (Sastrawi + NLTK) -----
stop_factory = StopWordRemoverFactory()
sastrawi_stopwords = set(stop_factory.get_stop_words())

# Gabungkan dengan stopwords NLTK (Bahasa Indonesia)
nltk_stopwords_id = set(stopwords.words('indonesian'))
combined_stopwords = sastrawi_stopwords.union(nltk_stopwords_id)

# Tambahkan stopwords custom yang umum di review aplikasi Indonesia
custom_stopwords = {
    'yg', 'dgn', 'utk', 'krn', 'sdh', 'udh', 'gak', 'ga', 'gk',
    'nya', 'aja', 'deh', 'sih', 'dong', 'loh', 'kak', 'mas', 'mba',
    'bgt', 'banget', 'juga', 'udah', 'sudah', 'bisa', 'tidak', 'tidak'
}
combined_stopwords = combined_stopwords.union(custom_stopwords)

print(f'Total stopwords: {len(combined_stopwords)}')

Total stopwords: 834


In [9]:
# ----- Kamus Slang / Normalisasi kata informal Bahasa Indonesia -----
# Kamus ini berisi kata-kata tidak baku yang umum digunakan di review
slang_dict = {
    # Kontraksi umum
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "ngga": "tidak",
    "nggak": "tidak", "enggak": "tidak", "tak": "tidak",
    "udah": "sudah", "udh": "sudah", "sdh": "sudah",
    "emang": "memang", "emg": "memang",
    "kalo": "kalau", "klo": "kalau",
    "yg": "yang", "yng": "yang",
    "dgn": "dengan", "dg": "dengan",
    "utk": "untuk", "buat": "untuk",
    "krn": "karena", "karna": "karena",
    "tapi": "tetapi", "tp": "tetapi",
    "jd": "jadi", "jdnya": "jadinya",
    "gimana": "bagaimana", "gmn": "bagaimana",
    "bgt": "banget", "bngt": "banget",
    "bnyk": "banyak", "byk": "banyak",
    "sm": "sama", "sma": "sama",
    "ny": "nya", "x": "",
    "lg": "lagi", "lgi": "lagi",
    "skrg": "sekarang", "skrang": "sekarang",
    "bs": "bisa", "bsa": "bisa",
    "dr": "dari", "dri": "dari",
    "sy": "saya", "aku": "saya",
    "td": "tadi", "tdi": "tadi",
    "blm": "belum", "blum": "belum",
    "msh": "masih", "msih": "masih",
    "bln": "bulan",
    "thn": "tahun",
    "hrg": "harga",
    "tmn": "teman",
    "hpus": "hapus",
    "downlod": "download",
    "instal": "install",
    "apk": "aplikasi",
    "app": "aplikasi",
    "hp": "handphone",
    "paket": "paket",
    "kuota": "kuota",
    "sinyal": "sinyal",
    "lemot": "lambat",
    "loadingnya": "loading",
    "error": "error",
    "ok": "oke",
    "okelah": "oke",
    "mantap": "bagus",
    "keren": "bagus",
    "wkwk": "", "wkwkwk": "", "haha": "", "hehe": "", "hihi": "",
    "xixi": "", "kwkw": "",
    "dong": "", "sih": "", "deh": "", "loh": "", "lah": "",
    "nih": "", "nah": "",
}

print(f'Jumlah entri kamus slang: {len(slang_dict)}')

Jumlah entri kamus slang: 86


## Fungsi-Fungsi Preprocessing

In [10]:
# ============================================================
# STEP 1: Lowercasing
# ============================================================
def lowercase_text(text):
    """Mengubah semua teks menjadi huruf kecil."""
    if pd.isna(text):
        return ''
    return str(text).lower()

# ============================================================
# STEP 2: Punctuation Removal
# ============================================================
def remove_punctuation(text):
    """Menghapus tanda baca, angka, dan karakter khusus."""
    # Hapus URL
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Hapus mention dan hashtag
    text = re.sub(r'@\w+|#\w+', '', text)
    # Hapus angka
    text = re.sub(r'\d+', '', text)
    # Hapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Hapus karakter non-ASCII / emoji
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ============================================================
# STEP 6: Expand Contractions / Normalisasi Slang
# ============================================================
def expand_contractions(text, slang_dict):
    """Menormalisasi kata-kata slang/singkatan Bahasa Indonesia."""
    tokens = text.split()
    normalized = [slang_dict.get(word, word) for word in tokens]
    # Hapus token kosong (kata yang dimap ke string kosong)
    normalized = [w for w in normalized if w != '']
    return ' '.join(normalized)

# ============================================================
# STEP 4: Tokenization
# ============================================================
def tokenize_text(text):
    """Memecah teks menjadi token/kata."""
    tokens = word_tokenize(text)
    return tokens

# ============================================================
# STEP 3: Stopword Removal
# ============================================================
def remove_stopwords(tokens, stopwords_set):
    """Menghapus stopwords dari list token."""
    return [word for word in tokens if word not in stopwords_set and len(word) > 1]

# ============================================================
# STEP 5: Stemming (Sastrawi)
# ============================================================
def stem_tokens(tokens, stemmer):
    """Melakukan stemming pada setiap token menggunakan Sastrawi."""
    return [stemmer.stem(word) for word in tokens]

# ============================================================
# STEP 7: Spelling Correction (opsional, mahal secara komputasi)
# ============================================================
def correct_spelling(tokens, spell):
    """
    Mengoreksi ejaan menggunakan SpellChecker.
    Catatan: SpellChecker berbasis English, untuk Bahasa Indonesia
    hasilnya terbatas. Gunakan sebagai langkah tambahan setelah normalisasi slang.
    """
    corrected = []
    for word in tokens:
        correction = spell.correction(word)
        corrected.append(correction if correction else word)
    return corrected

print('Semua fungsi preprocessing berhasil didefinisikan!')

Semua fungsi preprocessing berhasil didefinisikan!


## Preprocessing

In [11]:
# Salin dataframe untuk preprocessing
df_preprocessed = df_clean.copy()

print('Contoh teks asli:')
for i in range(3):
    print(f'  [{i}] {df_preprocessed["content"].iloc[i]}')

Contoh teks asli:
  [0] MAKIN HARI BUKANNYA TAMBAH BAIK MALAH TAMBAH BURUKKK. MENDING BUBARIN AJA XL PELAYANAN MAKIN BURUK.. SINYAL LELET.. GILIRAN HARGA DINAIKIN GA KIRA"
  [1] signal jatingan buruk & harga paket internet mahal jaringan buruk .
  [2] pelayanan busuk.. masa daftar kuota di apk jadi ilang.. xtra combo jadi gaada bebas puas gaada.. BANGKRUT AJA SEKALIAN.


In [12]:
# STEP 1: Lowercasing
print('Step 1: Lowercasing...')
df_preprocessed['step1_lowercase'] = df_preprocessed['content'].progress_apply(lowercase_text)

print('\nContoh setelah lowercase:')
for i in range(3):
    print(f'  [{i}] {df_preprocessed["step1_lowercase"].iloc[i]}')

Step 1: Lowercasing...


100%|██████████| 1006266/1006266 [00:02<00:00, 454303.37it/s]


Contoh setelah lowercase:
  [0] makin hari bukannya tambah baik malah tambah burukkk. mending bubarin aja xl pelayanan makin buruk.. sinyal lelet.. giliran harga dinaikin ga kira"
  [1] signal jatingan buruk & harga paket internet mahal jaringan buruk .
  [2] pelayanan busuk.. masa daftar kuota di apk jadi ilang.. xtra combo jadi gaada bebas puas gaada.. bangkrut aja sekalian.


In [13]:
# STEP 2: Punctuation Removal
print('Step 2: Punctuation Removal...')
df_preprocessed['step2_clean'] = df_preprocessed['step1_lowercase'].progress_apply(remove_punctuation)

print('\nContoh setelah punctuation removal:')
for i in range(3):
    print(f'  [{i}] {df_preprocessed["step2_clean"].iloc[i]}')

Step 2: Punctuation Removal...


100%|██████████| 1006266/1006266 [00:22<00:00, 44579.19it/s]



Contoh setelah punctuation removal:
  [0] makin hari bukannya tambah baik malah tambah burukkk mending bubarin aja xl pelayanan makin buruk sinyal lelet giliran harga dinaikin ga kira
  [1] signal jatingan buruk harga paket internet mahal jaringan buruk
  [2] pelayanan busuk masa daftar kuota di apk jadi ilang xtra combo jadi gaada bebas puas gaada bangkrut aja sekalian


In [14]:
# STEP 6: Expand Contractions
print('Step 6: Expand Contractions / Normalisasi Slang...')
df_preprocessed['step3_expanded'] = df_preprocessed['step2_clean'].progress_apply(
    lambda x: expand_contractions(x, slang_dict)
)

print('\nContoh setelah normalisasi slang:')
for i in range(3):
    print(f'  [{i}] {df_preprocessed["step3_expanded"].iloc[i]}')

Step 6: Expand Contractions / Normalisasi Slang...


100%|██████████| 1006266/1006266 [00:05<00:00, 169098.20it/s]



Contoh setelah normalisasi slang:
  [0] makin hari bukannya tambah baik malah tambah burukkk mending bubarin aja xl pelayanan makin buruk sinyal lelet giliran harga dinaikin tidak kira
  [1] signal jatingan buruk harga paket internet mahal jaringan buruk
  [2] pelayanan busuk masa daftar kuota di aplikasi jadi ilang xtra combo jadi gaada bebas puas gaada bangkrut aja sekalian


In [15]:
# STEP 4: Tokenization
print('Step 4: Tokenization...')
df_preprocessed['step4_tokens'] = df_preprocessed['step3_expanded'].progress_apply(tokenize_text)

print('\nContoh setelah tokenisasi:')
for i in range(3):
    print(f'  [{i}] {df_preprocessed["step4_tokens"].iloc[i]}')

Step 4: Tokenization...


100%|██████████| 1006266/1006266 [01:08<00:00, 14624.52it/s]


Contoh setelah tokenisasi:
  [0] ['makin', 'hari', 'bukannya', 'tambah', 'baik', 'malah', 'tambah', 'burukkk', 'mending', 'bubarin', 'aja', 'xl', 'pelayanan', 'makin', 'buruk', 'sinyal', 'lelet', 'giliran', 'harga', 'dinaikin', 'tidak', 'kira']
  [1] ['signal', 'jatingan', 'buruk', 'harga', 'paket', 'internet', 'mahal', 'jaringan', 'buruk']
  [2] ['pelayanan', 'busuk', 'masa', 'daftar', 'kuota', 'di', 'aplikasi', 'jadi', 'ilang', 'xtra', 'combo', 'jadi', 'gaada', 'bebas', 'puas', 'gaada', 'bangkrut', 'aja', 'sekalian']


In [16]:
# STEP 3: Stopword Removal
print('Step 3: Stopword Removal...')
df_preprocessed['step5_no_stopwords'] = df_preprocessed['step4_tokens'].progress_apply(
    lambda tokens: remove_stopwords(tokens, combined_stopwords)
)

print('\nContoh setelah stopword removal:')
for i in range(3):
    print(f'  [{i}] {df_preprocessed["step5_no_stopwords"].iloc[i]}')

Step 3: Stopword Removal...


100%|██████████| 1006266/1006266 [00:04<00:00, 208587.25it/s]


Contoh setelah stopword removal:
  [0] ['burukkk', 'mending', 'bubarin', 'xl', 'pelayanan', 'buruk', 'sinyal', 'lelet', 'giliran', 'harga', 'dinaikin']
  [1] ['signal', 'jatingan', 'buruk', 'harga', 'paket', 'internet', 'mahal', 'jaringan', 'buruk']
  [2] ['pelayanan', 'busuk', 'daftar', 'kuota', 'aplikasi', 'ilang', 'xtra', 'combo', 'gaada', 'bebas', 'puas', 'gaada', 'bangkrut']


In [17]:
# STEP 5: Stemming (Sastrawi)
print('Step 5: Stemming (Sastrawi) - ini bisa memakan waktu beberapa menit...')
df_preprocessed['step6_stemmed'] = df_preprocessed['step5_no_stopwords'].progress_apply(
    lambda tokens: stem_tokens(tokens, stemmer)
)

print('\nContoh setelah stemming:')
for i in range(3):
    print(f'  [{i}] {df_preprocessed["step6_stemmed"].iloc[i]}')

Step 5: Stemming (Sastrawi) - ini bisa memakan waktu beberapa menit...


100%|██████████| 1006266/1006266 [01:42<00:00, 9841.41it/s]



Contoh setelah stemming:
  [0] ['burukkk', 'mending', 'bubarin', 'xl', 'layan', 'buruk', 'sinyal', 'lelet', 'gilir', 'harga', 'dinaikin']
  [1] ['signal', 'jatingan', 'buruk', 'harga', 'paket', 'internet', 'mahal', 'jaring', 'buruk']
  [2] ['layan', 'busuk', 'daftar', 'kuota', 'aplikasi', 'ilang', 'xtra', 'combo', 'gaada', 'bebas', 'puas', 'gaada', 'bangkrut']


In [20]:
import pandas as pd
from collections import Counter

# STEP 8 & 9: Rare Words Removal & Common Words Removal
print('Menghitung frekuensi kata...')

# Kumpulkan semua token
all_tokens = [word for tokens in df_preprocessed['step6_stemmed'] for word in tokens]
word_freq = Counter(all_tokens)
total_words = len(word_freq)

print(f'Total kata unik sebelum filtering: {total_words}')

# Tampilkan distribusi frekuensi
freq_df = pd.DataFrame(word_freq.most_common(20), columns=['word', 'freq'])
print('\n20 Kata Paling Sering Muncul:')
print(freq_df.to_string(index=False))

Menghitung frekuensi kata...
Total kata unik sebelum filtering: 189375

20 Kata Paling Sering Muncul:
    word   freq
      xl 232430
aplikasi 189844
   bagus 170698
   paket 138716
  jaring 111614
    beli 105604
   kuota 104176
  sinyal  92819
   pulsa  74087
    buka  72267
    baik  70647
  tolong  59249
     oke  51619
  lambat  49042
    pake  39660
  update  39495
   mahal  37750
      my  37662
    myxl  37038
   bonus  36578


In [23]:
# Tentukan threshold
RARE_THRESHOLD = 2       # kata yang muncul <= 2 kali dianggap rare
COMMON_THRESHOLD = 0.85  # kata yang muncul di > 85% dokumen dianggap terlalu umum

# Hitung kata rare
rare_words = {word for word, freq in word_freq.items() if freq <= RARE_THRESHOLD}
print(f'Jumlah rare words (frekuensi <= {RARE_THRESHOLD}): {len(rare_words)}')

# Hitung kata common (document frequency)
total_docs = len(df_preprocessed)
doc_freq = Counter()
for tokens in df_preprocessed['step6_stemmed']:
    for word in set(tokens):  # set agar tidak duplikat per dokumen
        doc_freq[word] += 1

common_words = {word for word, df_count in doc_freq.items()
                if df_count / total_docs > COMMON_THRESHOLD}
print(f'Jumlah common words (doc freq > {COMMON_THRESHOLD*100}%): {len(common_words)}')

if common_words:
    print(f'Contoh common words: {list(common_words)[:10]}')

Jumlah rare words (frekuensi <= 2): 153526
Jumlah common words (doc freq > 85.0%): 0


In [26]:
# STEP 8: Rare Words Removal
print('Step 8: Rare Words Removal...')
df_preprocessed['step8_no_rare'] = df_preprocessed['step7_spell_corrected'].progress_apply(
    lambda tokens: [word for word in tokens if word not in rare_words]
)

# STEP 9: Common Words Removal
print('Step 9: Common Words Removal...')
df_preprocessed['step9_no_common'] = df_preprocessed['step8_no_rare'].progress_apply(
    lambda tokens: [word for word in tokens if word not in common_words]
)

print('\nContoh hasil akhir preprocessing:')
for i in range(5):
    print(f'  [{i}] {df_preprocessed["step9_no_common"].iloc[i]}')

Step 8: Rare Words Removal...


100%|██████████| 1006266/1006266 [00:05<00:00, 199051.18it/s]


Step 9: Common Words Removal...


100%|██████████| 1006266/1006266 [00:01<00:00, 565936.09it/s]



Contoh hasil akhir preprocessing:
  [0] ['burukkk', 'mending', 'bubarin', 'xl', 'layan', 'buruk', 'sinyal', 'lelet', 'gilir', 'harga', 'dinaikin']
  [1] ['signal', 'jatingan', 'buruk', 'harga', 'paket', 'internet', 'mahal', 'jaring', 'buruk']
  [2] ['layan', 'busuk', 'daftar', 'kuota', 'aplikasi', 'ilang', 'xtra', 'combo', 'gaada', 'bebas', 'puas', 'gaada', 'bangkrut']
  [3] ['busuk', 'ngeclaim', 'kuota', 'aktivasi', 'gabisa', 'aplikasi', 'hubung', 'cs', 'loading', 'doang', 'live', 'agent', 'parah', 'layan']
  [4] ['tolong', 'min', 'tempat', 'ku', 'sinyal', 'full', 'kuota', 'dipake']


gabungkan token menjadi teks & buat kolom final

In [27]:
# Gabungkan token kembali menjadi string
df_preprocessed['processed_text'] = df_preprocessed['step9_no_common'].apply(
    lambda tokens: ' '.join(tokens)
)

# Hapus baris yang teks processed-nya kosong
before = len(df_preprocessed)
df_preprocessed = df_preprocessed[df_preprocessed['processed_text'].str.strip() != '']
after = len(df_preprocessed)
print(f'Baris dihapus karena teks kosong setelah preprocessing: {before - after}')
print(f'Jumlah data final: {after}')

Baris dihapus karena teks kosong setelah preprocessing: 46889
Jumlah data final: 959377


In [28]:
# Buat dataframe hasil akhir
df_final = df_preprocessed[['content', 'processed_text', 'score']].copy()
df_final.reset_index(drop=True, inplace=True)

print('=== HASIL AKHIR PREPROCESSING ===')
print(f'Shape: {df_final.shape}')
df_final.head(10)

=== HASIL AKHIR PREPROCESSING ===
Shape: (959377, 3)


,content,processed_text,score
0,MAKIN HARI BUKANNYA TAMBAH BAIK MALAH TAMBAH B...,burukkk mending bubarin xl layan buruk sinyal ...,1
1,signal jatingan buruk & harga paket internet m...,signal jatingan buruk harga paket internet mah...,2
2,pelayanan busuk.. masa daftar kuota di apk jad...,layan busuk daftar kuota aplikasi ilang xtra c...,1
3,BUSUK KALI .. MAU NGECLAIM KUOTA AKTIVASI AJA ...,busuk ngeclaim kuota aktivasi gabisa aplikasi ...,1
4,"tolong min ditempat ku sinyal full, kuota bany...",tolong min tempat ku sinyal full kuota dipake,1
5,lelet,lelet,1
6,saya mau menambah kuota keluarga gagal terus ya,tambah kuota keluarga gagal,1
7,sinyalnya bapuk,sinyal bapuk,2
8,beli paket 8gb di aplikasi tp pas udh dibeli m...,beli paket gb aplikasi pas beli gb,1
9,aduh kurang niat xl niiii bintang 0 ni kalau b...,aduh niat xl niiii bintang ni bisaaaa jelassss,1


In [29]:
# Perbandingan sebelum dan sesudah preprocessing
print('Perbandingan teks asli vs hasil preprocessing:\n')
for i in range(5):
    print(f'[{i}] ASLI     : {df_final["content"].iloc[i]}')
    print(f'[{i}] PROSES   : {df_final["processed_text"].iloc[i]}')
    print()

Perbandingan teks asli vs hasil preprocessing:

[0] ASLI     : MAKIN HARI BUKANNYA TAMBAH BAIK MALAH TAMBAH BURUKKK. MENDING BUBARIN AJA XL PELAYANAN MAKIN BURUK.. SINYAL LELET.. GILIRAN HARGA DINAIKIN GA KIRA"
[0] PROSES   : burukkk mending bubarin xl layan buruk sinyal lelet gilir harga dinaikin

[1] ASLI     : signal jatingan buruk & harga paket internet mahal jaringan buruk .
[1] PROSES   : signal jatingan buruk harga paket internet mahal jaring buruk

[2] ASLI     : pelayanan busuk.. masa daftar kuota di apk jadi ilang.. xtra combo jadi gaada bebas puas gaada.. BANGKRUT AJA SEKALIAN.
[2] PROSES   : layan busuk daftar kuota aplikasi ilang xtra combo gaada bebas puas gaada bangkrut

[3] ASLI     : BUSUK KALI .. MAU NGECLAIM KUOTA AKTIVASI AJA GABISA LOH DI APK.. UDAH HUBUNGI CS JUGA MALAH LOADING DOANG DI LIVE AGENT. PARAH KALI INI PELAYANANNYA
[3] PROSES   : busuk ngeclaim kuota aktivasi gabisa aplikasi hubung cs loading doang live agent parah layan

[4] ASLI     : tolong min ditem

simpan hasil preprocessing

In [30]:
# Simpan ke CSV
output_filename = 'df_myxl_preprocessed.csv'
df_final.to_csv(output_filename, index=False)
print(f'Dataset berhasil disimpan sebagai: {output_filename}')

# Download file ke lokal
files.download(output_filename)
print('File siap didownload!')

Dataset berhasil disimpan sebagai: df_myxl_preprocessed.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File siap didownload!
